# 01 — 数据下载

本笔记本负责获取三条数据线：
- **特征层**：Sentinel-2 ARD（通过 DEA Sandbox）
- **标签层**：NSW SEED RUSLE C 因子栅格
- **辅助层**：DEM、降雨数据

> **运行环境**：推荐在 DEA Sandbox（sandbox.dea.ga.gov.au）运行，
> 本地运行需先 `pip install -r requirements.txt`

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
from pathlib import Path

from src.data_utils import load_config, ensure_dir, print_array_info

# 加载配置
cfg = load_config('../configs/config.yaml')
print('配置加载成功')
print(f"研究区域: {cfg['region']['name']}")
print(f"时间范围: {cfg['time']['start']} → {cfg['time']['end']}")

## 1. 连接 DEA Sandbox / Datacube

In [ ]:
# DEA Sandbox 环境中已预配置 datacube
# 本地环境需要配置 ~/.datacube.conf

try:
    import datacube
    dc = datacube.Datacube(app='nsw-cfactor')
    print('Datacube 连接成功')
    
    # 列出可用产品
    products = dc.list_products()
    s2_products = products[products['name'].str.contains('s2', case=False)]
    print(f"\n找到 {len(s2_products)} 个 Sentinel-2 相关产品")
    print(s2_products[['name', 'description']].head(5))
    
except ImportError:
    print('[提示] datacube 未安装，跳过此步骤')
    print('请在 DEA Sandbox 环境中运行，或运行: pip install datacube')

## 2. 定义研究区域（使用测试小区域加速验证）

In [ ]:
# 使用配置文件中的测试区域（悉尼西部农业区，约200km×170km）
bbox_test = cfg['region']['bbox_test']

# 研究区域参数
lon_min = bbox_test['lon_min']
lon_max = bbox_test['lon_max']
lat_min = bbox_test['lat_min']
lat_max = bbox_test['lat_max']

print(f"测试区域:")
print(f"  经度: {lon_min}°E → {lon_max}°E")
print(f"  纬度: {lat_min}°S → {lat_max}°S")

# 加载一个月的数据用于测试
TIME_START = '2022-01-01'
TIME_END   = '2022-01-31'
print(f"\n测试时间段: {TIME_START} → {TIME_END}")

## 3. 加载 Sentinel-2 ARD 数据（DEA Sandbox 路径）

In [ ]:
try:
    import datacube
    dc = datacube.Datacube(app='nsw-cfactor')
    
    # DEA 标准加载方式
    ds = dc.load(
        product=cfg['sentinel2']['collection'],
        x=(lon_min, lon_max),
        y=(lat_min, lat_max),
        time=(TIME_START, TIME_END),
        measurements=cfg['sentinel2']['bands'],
        output_crs=cfg['region']['crs'],
        resolution=(-cfg['sentinel2']['resolution'],
                     cfg['sentinel2']['resolution']),
    )
    
    print('Sentinel-2 数据加载成功')
    print(f'维度: {dict(ds.dims)}')
    print(f'波段: {list(ds.data_vars)}')
    print(ds)
    
except Exception as e:
    print(f'[跳过] DEA Datacube 不可用: {e}')
    print('\n备选方案：通过 STAC API 访问 Sentinel-2')
    print('>>> 见下方 STAC 单元格')

## 4. 备选：通过 STAC API 加载（本地环境）

In [ ]:
# 如果 DEA Datacube 不可用，使用 STAC API 访问 DEA 数据
# 需要: pip install pystac-client stackstac

try:
    from pystac_client import Client
    import stackstac
    
    DEA_STAC = 'https://explorer.dea.ga.gov.au/stac'
    catalog = Client.open(DEA_STAC)
    
    search = catalog.search(
        collections=['ga_s2_ard_granule_nrt'],
        bbox=[lon_min, lat_min, lon_max, lat_max],
        datetime=f"{TIME_START}/{TIME_END}",
        query={'eo:cloud_cover': {'lt': cfg['sentinel2']['cloud_cover_max']}}
    )
    
    items = list(search.get_items())
    print(f'找到 {len(items)} 个 Sentinel-2 场景')
    for item in items[:3]:
        print(f"  {item.id} | 云量: {item.properties.get('eo:cloud_cover', 'N/A')}%")

except ImportError:
    print('[提示] 请安装: pip install pystac-client stackstac')
except Exception as e:
    print(f'[错误] {e}')

## 5. 下载 NSW SEED 标签数据

In [ ]:
import requests
from src.data_utils import ensure_dir

raw_dir = ensure_dir('../data/raw')
labels_dir = ensure_dir('../data/raw/labels')

print('标签数据下载说明:')
print('='*50)
print('\n1. RUSLE C 因子栅格（SEED 平台）:')
print('   URL:', cfg['labels']['c_factor_seed_url'])
print('   -> 手动登录下载，或使用 SEED OData API')

print('\n2. 月度裸土侵蚀 GeoTIFF（示例：2022年1月）:')
# 注意：实际下载需要从 SEED API 获取最新的 resource URL
# 以下 URL 仅为示例格式
example_url = (
    'https://datasets.seed.nsw.gov.au/dataset/'
    'db9a8486-7401-4213-99b8-66037fda1a9c/resource/'
    '46d0c11b-4dc9-4985-a90e-a0ad3985c101/download/ero_202401b.tif'
)
print(f'   示例 URL: {example_url}')
print('\n[操作] 请访问 SEED 平台获取实际下载链接后，修改上方 URL 并运行下载')

## 6. 保存元数据记录

In [ ]:
import json
from datetime import datetime

metadata = {
    'created_at': datetime.now().isoformat(),
    'region': cfg['region'],
    'time_range': cfg['time'],
    'data_sources': {
        'sentinel2': 'DEA Sandbox / STAC API',
        'c_factor_label': 'NSW SEED RUSLE Factors',
        'monthly_erosion': 'NSW SEED Monthly GeoTIFF',
        'fractional_cover': 'DEA ga_ls_fc_pc_cyear_3'
    },
    'notes': '阶段2：数据下载记录'
}

meta_path = labels_dir / 'download_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
    
print(f'元数据已保存至: {meta_path}')
print('\n✓ 笔记本 01 完成')
print('下一步: 运行 02_feature_engineering.ipynb')